# 19 · Production smoke: dtrack_local end to end

Test 6 of `docs/plans/prod_smoke_tests.md`. The whole backup plan (`dtrack_local.run_all`: preflight, Stage 1 counts and gate, Stage 2 pulls and statistics, report) runs on a single pair and a small window, with the Right side read through the SQL warehouse, and the run folder backed up to an S3 scratch prefix.

Read-only to production: the run folder lives under `WORKDIR`, the backup goes to the scratch prefix, and both are removed at the end unless `KEEP=True`. No report store publish (`report_s3` stays `None`). With several pairs, each gets its own run; a failing pair is recorded and the loop moves on.

## How to approach

- **Prerequisites**: kernel = the repo `.venv`; `uv sync --extra dbx` plus `--extra athena` / `--extra ldap` if needed. `~/.databrickscfg` profile, `WAREHOUSE_ID`, Left credentials in the project `.env`, AWS credentials for `S3_SCRATCH`. Run notebooks 17 and 18 first: their failures are easier to read than a failed end-to-end run.
- **Run order**: top to bottom. The preflight cell is fast. The `run_all` cell is the slow one: it pulls every window of the pair from both sides, so keep `DAYS` small.
- **What to look at**: the summary table (`status`, findings, `dead`, `report_exists`, backup file count), then the per-run detail for row-check gates, col-check verdicts and dead-unit errors. Open `report/index.html` in a browser.
- **Failure modes**: `RuntimeError: preflight failed` = see the preflight table; status `PARTIAL` with no col-check = the Stage 1 gate held the pair back (large count differences; see `gate`), or units died; `dead > 0` = read the dead-unit errors; S3 errors in backup = scratch prefix permissions.

In [ ]:
import os
from pathlib import Path

CONFIG_PATH = os.environ.get("DTRACK_PROD_CONFIG", "C:/work/dtrack/c.json")   # your real pairs config
ENV_FILE = os.environ.get("DTRACK_ENV_FILE", "C:/work/dtrack/.env")             # project .env with <macro>_USR / _PWD / _DSN
PAIR = os.environ.get("DTRACK_NB_PAIR", "orders")                           # used when PAIRS = [PAIR]
PAIRS = [PAIR]                   # one pair by default; None = every active (non-skip) pair (slow: one full run each)
TO_DATE = os.environ.get("DTRACK_PROD_TO") or None      # None = yesterday; or "YYYY-MM-DD"
FROM_DATE = os.environ.get("DTRACK_PROD_FROM") or None  # None = TO_DATE - (DAYS - 1)
DAYS = 2                         # keep the window small: every window is pulled from both sides
GATE = "auto"                    # Stage 1 gate: "auto" | "manual" | "ratio:x"
S3_SCRATCH = os.environ.get("DTRACK_PROD_S3", "s3://<bucket>/dtrack/nb-scratch")   # backup target, removed at the end
DBX_PROFILE = os.environ.get("DTRACK_DBX_PROFILE", "corp")             # ~/.databrickscfg profile
WAREHOUSE_ID = os.environ.get("DTRACK_DBX_WAREHOUSE_ID", "")           # SQL warehouse id (required)
DISPOSITION = os.environ.get("DTRACK_LOCAL_DISPOSITION", "external_links")   # or "inline" when presigned downloads are blocked
INCLUDE_DATA = False             # True = also back up _local/data.duckdb (pulled production rows)
WORKDIR = Path.home() / ".local/plans/nbs/_work/19"     # run folders: WORKDIR/runs/<run_id>/
KEEP = False                     # True = keep the run folders and the S3 backup

In [ ]:
# Load the project .env first: load_config() only searches from the current
# directory, and this notebook does not run from the repo root.
import copy
import json
import traceback
from datetime import date, timedelta

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from dtrack_left.cli.config import load_config

pd.set_option("display.max_colwidth", 120)
if Path(ENV_FILE).exists():
    load_dotenv(ENV_FILE, override=False)
else:
    print(f"warning: {ENV_FILE} not found; Left credentials must already be in the environment")

# The real config: connection profiles resolved, col_map files expanded, validated.
config = load_config(CONFIG_PATH)

# Which pairs: every active (non-skip) pair, or the PAIRS list.
active = [name for name, spec in config["pairs"].items() if not spec.get("skip")]
if PAIRS is None:
    names = active
else:
    unknown = [name for name in PAIRS if name not in config["pairs"]]
    skipped = [name for name in PAIRS if name in config["pairs"] and name not in active]
    if unknown:
        print("not in the config, ignored:", unknown)
    if skipped:
        print("marked skip in the config, ignored:", skipped)
    names = [name for name in PAIRS if name in active]

# The check window: the last DAYS days up to TO_DATE (default yesterday).
TO = TO_DATE or (date.today() - timedelta(days=1)).isoformat()
FROM = FROM_DATE or (date.fromisoformat(TO) - timedelta(days=DAYS - 1)).isoformat()


# The pair exactly as configured, with fromDate/toDate narrowed to the window
# (a deep copy, so the loaded config is never changed).
def windowed(name):
    spec = copy.deepcopy(config["pairs"][name])
    spec["fromDate"], spec["toDate"] = FROM, TO
    return spec


WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"window {FROM}..{TO}; {len(names)} pair(s): {names}")

# Fresh run ids per session (open_run resumes an existing folder of the same id).
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s", force=True)
TAG = datetime.now().strftime("nb19-%Y%m%d-%H%M%S")
SCRATCH = f"{S3_SCRATCH.rstrip('/')}/{TAG}"
RUNS = WORKDIR / "runs"
print("scratch prefix:", SCRATCH)

## Connect to the SQL warehouse

One `Warehouse` shared by every run, so the Databricks login happens once.

In [ ]:
# One WorkspaceClient from ~/.databrickscfg (external-browser profiles open a
# login tab here), and the SQL warehouse read through the Statement Execution API.
from dtrack_left.backends.databricks import DEFAULT_PROFILE, workspace
from dtrack_local import Warehouse

client = workspace(DBX_PROFILE or DEFAULT_PROFILE)
warehouse = Warehouse(client, WAREHOUSE_ID, disposition=DISPOSITION)

# A trivial read proves auth, the warehouse id and the result disposition.
print(warehouse.rows("SELECT current_user(), current_catalog(), current_date()"))

## Open one local run per pair

`pipeline.open_run(config, run_id=..., out=..., target_factory=...)` creates the run folder (laid out like an S3 run) and its landing DuckDB. The config holds only this pair, narrowed to the window. The Right side is a `WarehouseTarget` on the shared warehouse that also lands each pulled Right window in the run's DuckDB.

In [ ]:
# The factory needs the run's landing DuckDB, which exists only after open_run;
# `holder` closes over the run object so the factory can reach it.
from dtrack_local import pipeline

RUNS_BY_PAIR = {}
for name in names:
    holder = {}
    document = {**config, "pairs": {name: windowed(name)}}

    def factory(spec, holder=holder):
        run = holder["run"]
        return pipeline.WarehouseTarget(warehouse, spec, run.local / "right", landing=run.landing)

    run = pipeline.open_run(document, run_id=f"{TAG}-{name}", out=RUNS, target_factory=factory)
    holder["run"] = run
    RUNS_BY_PAIR[name] = run
    print(f"{name}: {run.folder}")

## Preflight both sides

`pipeline.preflight(run)`: Left `SELECT ... WHERE 1=0`, Right `DESCRIBE QUERY`. `run_all` does the same and stops on failure; running it here first gives a readable table.

In [ ]:
rows = []
for name, run in RUNS_BY_PAIR.items():
    try:
        health = pipeline.preflight(run).get(name, {})
        rows.append({"pair": name,
                     "left_ok": health.get("left", {}).get("ok"), "left_error": health.get("left", {}).get("error", ""),
                     "right_ok": health.get("right", {}).get("ok"), "right_error": health.get("right", {}).get("error", "")})
    except Exception as exc:
        rows.append({"pair": name, "left_ok": False, "left_error": repr(exc)})
display(pd.DataFrame(rows))

## Run the whole plan

`pipeline.run_all(run, gate=GATE, backup_s3=...)`: preflight, Stage 1, Stage 2 (only when the gate closed Stage 1), report into `<run>/report/`, then the run folder (without `_local/`) copied to `<scratch>/runs/<run_id>/`. Exceptions are caught per pair.

In [ ]:
OUTCOMES, ERRORS = {}, {}
for name, run in RUNS_BY_PAIR.items():
    print(f"=== {name} ({FROM}..{TO})")
    try:
        OUTCOMES[name] = pipeline.run_all(run, gate=GATE, backup_s3=f"{SCRATCH}/runs", include_data=INCLUDE_DATA)
        print(f"{name}: {OUTCOMES[name]['status']} {OUTCOMES[name]['findings']}")
    except Exception as exc:
        ERRORS[name] = traceback.format_exc()
        print(f"{name}: FAILED {exc!r}")

## Summary per pair

Status (`DONE` / `PARTIAL`), the three finding counts, dead units, whether `report/index.html` exists, and what the backup uploaded.

In [ ]:
rows = []
for name, run in RUNS_BY_PAIR.items():
    outcome = OUTCOMES.get(name)
    index = run.folder / "report" / "index.html"
    row = {"pair": name, "status": outcome["status"] if outcome else "ERROR",
           "dead": len(run.store.list_keys("dead/")), "report_exists": index.exists(), "report": index.as_uri() if index.exists() else ""}
    if outcome:
        row.update(outcome["findings"])
        backup = outcome.get("backup") or {}
        row.update({"stage2_ran": outcome["col_check"] is not None, "backup_files": backup.get("files"),
                    "backup_uri": backup.get("uri")})
    else:
        row["error"] = (ERRORS.get(name, "").strip().splitlines() or [""])[-1]
    row["ok"] = bool(outcome) and row["status"] in {"DONE", "PARTIAL"} and row["dead"] == 0 and row["report_exists"]
    rows.append(row)
summary = pd.DataFrame(rows)
display(summary)

## Look inside each run

Stage 1 per-pair summary and gate decision, Stage 2 verdict counts, the error of every dead unit, the landing DuckDB tables, and the report path. Full tracebacks for failed pairs are printed last.

In [ ]:
for name, run in RUNS_BY_PAIR.items():
    outcome = OUTCOMES.get(name)
    print(f"=============== {name}: {run.folder}")
    if outcome:
        print("row_check:", json.dumps(outcome["row_check"], indent=2, default=str))
        print("col_check:", json.dumps(outcome["col_check"], indent=2, default=str))
        print("landing tables:", outcome["landing"])
        print("status file:", run.status())
    dead = run.store.list_keys("dead/")
    if dead:
        display(pd.DataFrame([{"key": key, **{k: str(v)[:200] for k, v in run.store.get_json(key).items()}}
                              for key in dead if key.endswith(".json")]))
    index = run.folder / "report" / "index.html"
    print("report:", index.as_uri() if index.exists() else "not built")
for name, text in ERRORS.items():
    print(f"=============== {name} traceback\n{text}")

In [ ]:
# The report's own summary.json (same numbers the HTML report shows).
for name, run in RUNS_BY_PAIR.items():
    path = run.folder / "report" / "summary.json"
    if path.exists():
        print(f"--- {name}")
        print(json.dumps(json.loads(path.read_text()), indent=2)[:3000])

## Cleanup

Removes the S3 backup under the scratch prefix and the local run folders (they hold pulled production rows) unless `KEEP=True`. Open the reports before running this.

In [ ]:
# Delete everything under the scratch prefix unless KEEP=True. This is the only
# S3 location the notebook ever writes to.
import shutil

from dtrack_left.backends.store import S3Store

if KEEP:
    print("KEEP=True: left in place:", SCRATCH, "and", WORKDIR)
else:
    store = S3Store(SCRATCH)
    keys = store.list_keys("")
    for start in range(0, len(keys), 1000):
        batch = [{"Key": store._key(key)} for key in keys[start : start + 1000]]
        store.client.delete_objects(Bucket=store.bucket, Delete={"Objects": batch})
    print(f"deleted {len(keys)} object(s) under {SCRATCH}")
    shutil.rmtree(RUNS, ignore_errors=True)
    print("removed", RUNS)

## What to check

- Preflight: both sides `ok=True`.
- Summary: `status` is `DONE` or `PARTIAL`, `dead == 0`, `report_exists`, `backup_files > 0`; `ok=True` sums these up.
- `stage2_ran=False` means the Stage 1 gate held the pair back: look at `row_check` gate and the count differences (notebook 18, test 5).
- Findings (`row_count_diff`, `partition_missing`, `col_stat_diff`) are real-data results, not failures; open the report to read them.